<a href="https://colab.research.google.com/github/heyanugrah/pytorch/blob/main/basicTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Tokenized the input text "I am Anugrah" into numerical IDs.

Embedded these token IDs into d_model (8-dimensional) vectors.

Added Positional Encodings to the embeddings to introduce sequence order information.

Calculated the Query (Q), Key (K), and Value (V) matrices from the embedded input using separate linear layers.

Split Q, K, and V into multiple attention heads (2 heads in this case).
Computed attention scores by multiplying Q and K.transpose(), then scaled these scores by sqrt(head_dim).

Applied a softmax function to get attention probabilities.

Multiplied these attention probabilities by the V matrix to get the out tensor for each head.

Combined the outputs from the multiple attention heads back into a single tensor.

Applied a final linear projection (Wo) to the combined heads, resulting in the output tensor, which has the same shape as the original embedded input (batch_size, seq_len, d_model).

In [278]:

#Create Vocabulary
vocab = {
    "I":0,
    "am":1,
    "Anugrah":2,
    "boy":3
}

In [279]:
vocab_size = len(new_vocab)
vocab_size

37

In [280]:
import torch

def tokenize(input_text, vocab, max_seq_len):
    token_ids = [vocab.get("<cls>")] # Start with the <cls> token
    for word in input_text.lower().replace('.', '').split():
        token_ids.append(vocab.get(word, vocab["<unk>"])) # Map unknown words to <unk> token

    # Truncate if sequence is too long
    if len(token_ids) > max_seq_len:
        token_ids = token_ids[:max_seq_len]
    # Pad if sequence is too short
    while len(token_ids) < max_seq_len:
        token_ids.append(vocab["<pad>"])

    # Convert to PyTorch tensor and add the batch dimension
    tensor_ids = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0)

    return tensor_ids

In [281]:
# Define a maximum sequence length for all inputs
max_seq_len = 10 # Adjust based on your dataset's typical sentence length

# Create Input and tokenize using the updated function and new vocabulary
x = tokenize("hello world this is another sentence", new_vocab, max_seq_len)
print("Tokenized input 'x':", x)
print("Shape of tokenized input 'x':", x.shape)

Tokenized input 'x': tensor([[2, 1, 1, 3, 4, 1, 7, 0, 0, 0]])
Shape of tokenized input 'x': torch.Size([1, 10])


In [282]:
print(x.shape)

torch.Size([1, 10])


In [283]:
#Embeding layer
#Transformers cannot work directly on integers.So we convert them into vectors.

import torch.nn as nn
d_model = 8

embedding = nn.Embedding(
    num_embeddings=vocab_size, # Use the updated vocab_size
    embedding_dim=d_model
)

In [284]:
embedded = embedding(x)
print(embedded.shape)

torch.Size([1, 10, 8])


In [285]:
print(embedded)

tensor([[[-0.0467, -0.5198,  0.2803,  1.9535, -2.5634,  1.5412, -0.6248,
           0.7039],
         [ 0.3641, -0.4808,  0.0204,  0.5036,  0.2052, -1.2413, -0.2396,
           1.4316],
         [ 0.3641, -0.4808,  0.0204,  0.5036,  0.2052, -1.2413, -0.2396,
           1.4316],
         [ 0.7075,  0.1216, -1.2115,  0.5131, -1.3885, -0.8683,  0.7760,
          -0.4265],
         [-1.0084,  0.4919,  0.1223,  0.8691,  0.5494, -1.4269,  0.6356,
           1.4650],
         [ 0.3641, -0.4808,  0.0204,  0.5036,  0.2052, -1.2413, -0.2396,
           1.4316],
         [-1.4355, -0.2604, -2.0536, -0.0265, -0.0438,  0.8311, -0.7982,
           0.2032],
         [-0.0204, -3.0104,  1.1661, -0.7317,  0.3679,  0.0726,  0.7143,
          -0.5669],
         [-0.0204, -3.0104,  1.1661, -0.7317,  0.3679,  0.0726,  0.7143,
          -0.5669],
         [-0.0204, -3.0104,  1.1661, -0.7317,  0.3679,  0.0726,  0.7143,
          -0.5669]]], grad_fn=<EmbeddingBackward0>)


In [286]:
#Create Positional Encoding
#Transformer doesn't know order.

In [287]:
import math

In [288]:
import math

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=100):
        super().__init__()

        pe = torch.zeros(max_len, d_model) # making zero matrix
        position = torch.arange(0,max_len).unsqueeze(1) #making column vector for 100 - two dimensional
        div_term = torch.exp(torch.arange(0,d_model,2) *(-math.log(10000.0) / d_model))
        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos( position * div_term)

        '''
          #Embedding dimensions:

           0 1 2 3 4 5 6 7
           ↑   ↑   ↑   ↑
          sin sin sin sin
          -------------------
          0 1 2 3 4 5 6 7
            ↑   ↑   ↑   ↑
           cos cos cos cos

          Final positional encoding pattern:
          [sin, cos, sin, cos, sin, cos, sin, cos]
          '''
        self.register_buffer("pe",pe) #pe belongs to the model ,but pe is NOT trainable

    def forward(self, x):
          seq_len = x.size(1)
          return x + self.pe[:seq_len]


In [289]:
pos_encoder = PositionalEncoding(d_model)

In [290]:
pos_encoder

PositionalEncoding()

In [291]:
embedded = pos_encoder(embedded)
print(embedded.shape)

torch.Size([1, 10, 8])


In [292]:
embedded

tensor([[[-0.0467,  0.4802,  0.2803,  2.9535, -2.5634,  2.5412, -0.6248,
           1.7039],
         [ 1.2056,  0.0595,  0.1202,  1.4986,  0.2152, -0.2414, -0.2386,
           2.4316],
         [ 1.2734, -0.8970,  0.2191,  1.4837,  0.2252, -0.2415, -0.2376,
           2.4316],
         [ 0.8486, -0.8683, -0.9160,  1.4685, -1.3585,  0.1312,  0.7790,
           0.5735],
         [-1.7652, -0.1618,  0.5117,  1.7901,  0.5894, -0.4277,  0.6396,
           2.4649],
         [-0.5948, -0.1972,  0.4998,  1.3812,  0.2552, -0.2426, -0.2346,
           2.4316],
         [-1.7149,  0.6997, -1.4890,  0.7988,  0.0161,  1.8293, -0.7922,
           1.2032],
         [ 0.6366, -2.2565,  1.8103,  0.0331,  0.4378,  1.0702,  0.7213,
           0.4331],
         [ 0.9690, -3.1559,  1.8834, -0.0350,  0.4478,  1.0694,  0.7223,
           0.4331],
         [ 0.3917, -3.9215,  1.9494, -0.1101,  0.4578,  1.0686,  0.7233,
           0.4331]]], grad_fn=<AddBackward0>)

In [293]:
#Multi-head-attention

In [294]:
#Create Q, K, V

In [295]:
Wq = nn.Linear(8,8) #weighted query
Wk = nn.Linear(8,8) #weighted key
Wv = nn.Linear(8,8) #weighted value

In [296]:
#Split Into Multiple Heads
num_heads = 2 #attention heads
d_model = 8 #8 dimensional vector for word

In [297]:
embedded.shape

torch.Size([1, 10, 8])

In [298]:
embedded.shape[0] ,embedded.shape[1],embedded.shape[2] #batch_size,no of words,dimension of word vector

(1, 10, 8)

In [299]:
Q = Wq(embedded)
K = Wk(embedded)
V = Wv(embedded)

In [300]:
Q

tensor([[[-1.7771,  0.5319,  1.0875,  2.3951, -2.1055, -0.2936, -0.2989,
          -0.9330],
         [-0.3343,  1.0077,  0.3914,  0.9368, -0.8375, -0.1256,  0.6951,
          -0.4871],
         [-0.2927,  1.1822,  0.6306,  0.8702, -0.6215,  0.2098,  0.4865,
          -0.5458],
         [-0.7408,  0.4507,  0.9840,  0.4881, -1.2937, -0.6314, -0.0156,
           0.0739],
         [ 0.0975,  1.4057,  0.9426,  0.8872, -0.5745,  0.1079,  0.3477,
          -1.0295],
         [ 0.0854,  1.0892,  0.6427,  1.0458, -0.5286,  0.1602,  0.2814,
          -0.8800],
         [ 0.1444,  0.2881, -0.0915,  0.9410, -1.0632,  0.0250, -1.2135,
          -0.8199],
         [-0.3125,  1.5107,  1.2632,  0.4793,  0.3994,  1.2038, -0.4192,
          -0.5539],
         [-0.3130,  1.6659,  1.4613,  0.3892,  0.5891,  1.5054, -0.5724,
          -0.5607],
         [-0.1203,  1.7851,  1.6946,  0.3469,  0.8168,  1.8157, -0.9002,
          -0.7246]]], grad_fn=<ViewBackward0>)

In [301]:

batch_size = embedded.shape[0] #batch size
seq_len = embedded.shape[1] #no of words
head_dim = d_model // num_heads

'''
Why multiple heads?

Instead of one attention mechanism:

Head 1 → grammar
Head 2 → meaning

Each head learns different relationships.
'''

Q = Q.view(
    batch_size,
    seq_len,
    num_heads,
    head_dim
).transpose(1,2)

K = K.view(
    batch_size,
    seq_len,
    num_heads,
    head_dim
).transpose(1,2)

V = V.view(
    batch_size,
    seq_len,
    num_heads,
    head_dim
).transpose(1,2)

In [302]:
# Attention Scores

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$



Here is a quick breakdown of exactly what each piece represents in the equation:

$Q$: The Query matrix (what the token is looking for).

$K^T$: The transposed Key matrix (what the other tokens contain).

Multiplying $Q$ by $K^T$ calculates the raw attention scores.

 $\sqrt{d_k}$: The Scaling Factor. It is the square root of the key dimension ($d_k$), used to shrink the raw scores so the softmax function does not break.  

 $\text{softmax}$: The function that turns the scaled scores into probabilities that sum to 1.

$V$: The Value matrix. The probabilities are multiplied by $V$ to extract the final context-aware output.  

In [303]:
scores = torch.matmul(
    Q,
    K.transpose(-2,-1)
)

In [304]:
scores

tensor([[[[-1.1459,  1.3189,  1.5796, -0.1934,  0.7788,  0.9835, -1.0397,
            1.1787,  1.4832,  1.5015],
          [ 0.6488,  1.2155,  1.1401,  0.1706,  1.0616,  1.0507, -0.0482,
            0.6615,  0.6096,  0.4562],
          [ 1.0379,  1.2099,  1.1086,  0.1616,  0.6442,  0.8680, -0.3026,
            0.6798,  0.6289,  0.3823],
          [ 0.7480,  0.3270,  0.3335, -0.0808, -1.2605, -0.4148, -1.0156,
            0.2142,  0.3237,  0.0254],
          [ 1.0824,  1.0552,  0.9881,  0.0606,  0.4085,  0.7401, -0.7391,
            0.9305,  0.9203,  0.6883],
          [ 0.2703,  0.9555,  0.9690,  0.0128,  0.9945,  0.9605, -0.4887,
            0.9754,  1.0005,  0.9562],
          [-1.0791,  0.5213,  0.6472, -0.0486,  1.7597,  1.0894,  0.1387,
            0.7601,  0.8150,  1.0946],
          [ 2.2780,  1.0658,  0.8704,  0.1499, -0.9122,  0.1005, -0.9857,
            0.5041,  0.4482, -0.1125],
          [ 2.7041,  1.0816,  0.8499,  0.1599, -1.3175, -0.0742, -1.1778,
            0.4784,  0

In [305]:
#Scale

In [306]:
scores = scores / math.sqrt(head_dim)

In [307]:
scores


tensor([[[[-0.5730,  0.6594,  0.7898, -0.0967,  0.3894,  0.4918, -0.5199,
            0.5893,  0.7416,  0.7508],
          [ 0.3244,  0.6077,  0.5700,  0.0853,  0.5308,  0.5253, -0.0241,
            0.3308,  0.3048,  0.2281],
          [ 0.5190,  0.6049,  0.5543,  0.0808,  0.3221,  0.4340, -0.1513,
            0.3399,  0.3144,  0.1911],
          [ 0.3740,  0.1635,  0.1668, -0.0404, -0.6303, -0.2074, -0.5078,
            0.1071,  0.1618,  0.0127],
          [ 0.5412,  0.5276,  0.4941,  0.0303,  0.2043,  0.3701, -0.3695,
            0.4652,  0.4602,  0.3442],
          [ 0.1351,  0.4778,  0.4845,  0.0064,  0.4973,  0.4802, -0.2443,
            0.4877,  0.5003,  0.4781],
          [-0.5396,  0.2606,  0.3236, -0.0243,  0.8799,  0.5447,  0.0693,
            0.3800,  0.4075,  0.5473],
          [ 1.1390,  0.5329,  0.4352,  0.0750, -0.4561,  0.0502, -0.4929,
            0.2521,  0.2241, -0.0562],
          [ 1.3521,  0.5408,  0.4250,  0.0799, -0.6587, -0.0371, -0.5889,
            0.2392,  0

In [308]:
#Softmax

In [309]:
'''
Before:

[4,2,1]

After:

[0.84,0.11,0.05]

84% attention to token 1
11% attention to token 2
5% attention to token 3
'''

attention = torch.softmax(
    scores,
    dim=-1
)



In [310]:
attention

tensor([[[[0.0368, 0.1261, 0.1437, 0.0592, 0.0963, 0.1066, 0.0388, 0.1176,
           0.1369, 0.1382],
          [0.0957, 0.1271, 0.1224, 0.0754, 0.1177, 0.1170, 0.0676, 0.0963,
           0.0939, 0.0869],
          [0.1191, 0.1298, 0.1234, 0.0769, 0.0978, 0.1094, 0.0609, 0.0996,
           0.0971, 0.0858],
          [0.1450, 0.1175, 0.1179, 0.0958, 0.0531, 0.0811, 0.0601, 0.1111,
           0.1173, 0.1011],
          [0.1224, 0.1208, 0.1168, 0.0734, 0.0874, 0.1032, 0.0492, 0.1135,
           0.1129, 0.1005],
          [0.0799, 0.1126, 0.1133, 0.0703, 0.1148, 0.1129, 0.0547, 0.1137,
           0.1152, 0.1126],
          [0.0412, 0.0918, 0.0978, 0.0690, 0.1705, 0.1219, 0.0758, 0.1034,
           0.1063, 0.1223],
          [0.2361, 0.1288, 0.1168, 0.0815, 0.0479, 0.0795, 0.0462, 0.0973,
           0.0946, 0.0715],
          [0.2839, 0.1262, 0.1124, 0.0796, 0.0380, 0.0708, 0.0408, 0.0933,
           0.0903, 0.0648],
          [0.3025, 0.1185, 0.1063, 0.0773, 0.0323, 0.0647, 0.0349, 0.0984

In [311]:
#Multiply By Values

In [312]:
out = attention @ V

In [313]:
out

tensor([[[[-0.3154,  0.4262,  0.0417, -0.4619],
          [-0.3258,  0.0763,  0.0988, -0.4713],
          [-0.3352,  0.0839,  0.1035, -0.4588],
          [-0.4035,  0.2200,  0.1078, -0.3854],
          [-0.3578,  0.2004,  0.1033, -0.4307],
          [-0.3516,  0.2546,  0.0915, -0.4389],
          [-0.3669,  0.2374,  0.1038, -0.4371],
          [-0.3896, -0.0132,  0.1573, -0.4062],
          [-0.4084, -0.0759,  0.1846, -0.3905],
          [-0.4288, -0.0459,  0.1967, -0.3694]],

         [[-0.7498, -0.7505,  1.0724, -0.0306],
          [-0.7520, -0.7106,  1.0865, -0.0549],
          [-0.8566, -0.6797,  1.0199,  0.0224],
          [-0.5419, -0.7581,  1.1639, -0.1203],
          [-0.8573, -0.6806,  1.0012,  0.0777],
          [-0.8610, -0.6754,  0.9945,  0.0806],
          [-0.8079, -0.6990,  0.9290,  0.1950],
          [-0.9245, -0.5598,  0.8210,  0.4277],
          [-0.9058, -0.5345,  0.7886,  0.5301],
          [-0.8502, -0.5166,  0.7557,  0.6867]]]],
       grad_fn=<UnsafeViewBackward0

In [314]:
#Combine Heads

In [315]:
combined_heads = out.transpose(1,2).contiguous().view(batch_size, seq_len, d_model)

In [316]:
combined_heads

tensor([[[-0.3154,  0.4262,  0.0417, -0.4619, -0.7498, -0.7505,  1.0724,
          -0.0306],
         [-0.3258,  0.0763,  0.0988, -0.4713, -0.7520, -0.7106,  1.0865,
          -0.0549],
         [-0.3352,  0.0839,  0.1035, -0.4588, -0.8566, -0.6797,  1.0199,
           0.0224],
         [-0.4035,  0.2200,  0.1078, -0.3854, -0.5419, -0.7581,  1.1639,
          -0.1203],
         [-0.3578,  0.2004,  0.1033, -0.4307, -0.8573, -0.6806,  1.0012,
           0.0777],
         [-0.3516,  0.2546,  0.0915, -0.4389, -0.8610, -0.6754,  0.9945,
           0.0806],
         [-0.3669,  0.2374,  0.1038, -0.4371, -0.8079, -0.6990,  0.9290,
           0.1950],
         [-0.3896, -0.0132,  0.1573, -0.4062, -0.9245, -0.5598,  0.8210,
           0.4277],
         [-0.4084, -0.0759,  0.1846, -0.3905, -0.9058, -0.5345,  0.7886,
           0.5301],
         [-0.4288, -0.0459,  0.1967, -0.3694, -0.8502, -0.5166,  0.7557,
           0.6867]]], grad_fn=<ViewBackward0>)

In [317]:
Wo = nn.Linear(d_model, d_model) # Output Linear Layer

In [318]:
output = Wo(combined_heads)

Multi-Head Attention Output: The initial output from the attention mechanism has a shape of torch.Size([1, 3, 8]).

First Layer Normalization: This was applied to the Multi-Head Attention output, resulting in normalized_output_1 with the same shape torch.Size([1, 3, 8]). This normalization helps stabilize the network training.

Feed-Forward Network (FFN): The FFN processed normalized_output_1, expanding the dimensions and then projecting them back. The ffn_output also has a shape of torch.Size([1, 3, 8]).

Residual Connection (FFN): The output of the FFN (ffn_output) was added back to its input (normalized_output_1). This residual_ffn also maintains the shape torch.Size([1, 3, 8]) and helps with gradient flow during training.

Second Layer Normalization: Finally, another Layer Normalization was applied to the residual_ffn to produce final_block_output, which represents the output of this complete Transformer block segment, maintaining the shape torch.Size([1, 3, 8])

In [319]:
print(output.shape)
print(output)

# Next: Layer Normalization (after Multi-Head Attention)
layer_norm_1 = nn.LayerNorm(d_model)
normalized_output_1 = layer_norm_1(output)
print("\nShape after Layer Normalization (1):", normalized_output_1.shape)
print("Output after Layer Normalization (1):\n", normalized_output_1)

# Next: Feed-Forward Network
ffn = nn.Sequential(
    nn.Linear(d_model, 4 * d_model), # First linear layer expands dimension
    nn.ReLU(),                        # Non-linear activation
    nn.Linear(4 * d_model, d_model)   # Second linear layer projects back to d_model
)
ffn_output = ffn(normalized_output_1)
print("\nShape after Feed-Forward Network:", ffn_output.shape)
print("Output after Feed-Forward Network:\n", ffn_output)

# Next: Residual connection and Layer Normalization (after FFN)
# Add the FFN output to its input (after the first LayerNorm)
residual_ffn = normalized_output_1 + ffn_output
print("\nShape after Residual Connection (FFN):", residual_ffn.shape)
print("Output after Residual Connection (FFN):\n", residual_ffn)

layer_norm_2 = nn.LayerNorm(d_model)
final_block_output = layer_norm_2(residual_ffn)
print("\nShape after Layer Normalization (2):", final_block_output.shape)
print("Output after Layer Normalization (2):\n", final_block_output)

torch.Size([1, 10, 8])
tensor([[[-7.0481e-01,  5.4265e-02,  2.2825e-01, -1.9200e-01, -1.7353e-01,
          -4.3694e-01,  1.3934e-01,  2.3861e-02],
         [-6.0678e-01,  2.3415e-02,  3.1308e-01, -1.0897e-01, -2.8290e-02,
          -4.6844e-01,  1.3344e-01,  1.2548e-01],
         [-6.5596e-01,  2.2309e-02,  3.2622e-01, -9.6993e-02, -1.1148e-03,
          -4.0159e-01,  1.1321e-01,  1.2454e-01],
         [-5.6426e-01,  8.6848e-02,  2.9990e-01, -1.0586e-01, -1.9120e-01,
          -5.4143e-01,  1.6330e-01,  3.9846e-02],
         [-6.9421e-01,  3.6078e-02,  3.1191e-01, -1.0801e-01, -6.4658e-02,
          -3.7041e-01,  1.0820e-01,  8.5662e-02],
         [-7.1103e-01,  3.5149e-02,  2.9711e-01, -1.2337e-01, -7.9694e-02,
          -3.6603e-01,  1.0674e-01,  6.9931e-02],
         [-7.2210e-01,  1.0629e-02,  2.9913e-01, -1.0475e-01, -1.0784e-01,
          -3.3701e-01,  8.1967e-02,  1.0619e-01],
         [-7.2350e-01, -5.9659e-02,  3.8907e-01, -4.2844e-03,  1.3115e-03,
          -2.2231e-01,  2.5

### Encapsulating into a TransformerEncoderLayer

Now that you've built out each component step-by-step, let's combine them into a single, reusable `TransformerEncoderLayer` class. This class will perform the entire sequence of operations you've defined: Multi-Head Attention, Layer Normalization, Feed-Forward Network, and the subsequent residual connection and Layer Normalization. This makes it easy to stack multiple such layers to form a complete Transformer Encoder.

In [320]:
import torch
import torch.nn as nn
import math

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ffn_hidden_dim, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Multi-Head Attention components
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout_rate) # Dropout after MHA output

        # First Layer Normalization (applied to MHA output, before FFN input)
        self.layer_norm1 = nn.LayerNorm(d_model)

        # Feed-Forward Network components
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_hidden_dim), # First linear layer expands dimension
            nn.ReLU(),                        # Non-linear activation
            nn.Linear(ffn_hidden_dim, d_model)   # Second linear layer projects back to d_model
        )
        self.ffn_dropout = nn.Dropout(dropout_rate) # Dropout after FFN output

        # Second Layer Normalization (after FFN residual connection)
        self.layer_norm2 = nn.LayerNorm(d_model)

    def _calculate_attention(self, Q, K, V):
        # Scaled Dot-Product Attention
        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_dim)
        attention_weights = torch.softmax(scores, dim=-1)
        out = attention_weights @ V
        return out

    def forward(self, x):
        batch_size, seq_len, _ = x.shape # x is the input to the encoder layer (e.g., embedded)

        # --- Multi-Head Attention Sub-layer ---
        # 1. Linear projections for Q, K, V from input x
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # 2. Split into multiple heads and transpose
        Q_heads = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K_heads = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V_heads = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # 3. Calculate attention for each head
        attn_output_heads = self._calculate_attention(Q_heads, K_heads, V_heads)

        # 4. Combine heads and apply final linear projection (Wo)
        combined_heads = attn_output_heads.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        mha_output = self.Wo(combined_heads)
        mha_output = self.attn_dropout(mha_output) # Apply dropout after MHA output

        # 5. First Layer Normalization: applied directly to MHA output
        # This corresponds to 'normalized_output_1' in your previous steps.
        input_to_ffn_sublayer = self.layer_norm1(mha_output)

        # --- Feed-Forward Network Sub-layer ---
        ffn_output = self.ffn(input_to_ffn_sublayer)
        ffn_output = self.ffn_dropout(ffn_output) # Apply dropout after FFN output

        # --- Residual connection and Second Layer Normalization (after FFN) ---
        # Residual connection: input to FFN sublayer + FFN output
        # This corresponds to 'residual_ffn' in your previous steps.
        residual_sum_ffn = input_to_ffn_sublayer + ffn_output
        # Second Layer Normalization: applied to the residual sum
        # This corresponds to 'final_block_output' in your previous steps.
        final_output = self.layer_norm2(residual_sum_ffn)

        return final_output

In [321]:
# Demonstrate the TransformerEncoderLayer
ffn_hidden_dim = 4 * d_model # Common practice for FFN hidden dimension

encoder_layer = TransformerEncoderLayer(d_model=d_model, num_heads=num_heads, ffn_hidden_dim=ffn_hidden_dim)

# Pass the 'embedded' tensor (input to the block) through the new layer
final_block_output_from_class = encoder_layer(embedded)

print("\nShape of final output from TransformerEncoderLayer:", final_block_output_from_class.shape)
print("Final output from TransformerEncoderLayer:\n", final_block_output_from_class)


Shape of final output from TransformerEncoderLayer: torch.Size([1, 10, 8])
Final output from TransformerEncoderLayer:
 tensor([[[ 0.8714,  1.9511, -0.7676, -0.4113, -0.7367,  0.5945, -1.3218,
          -0.1796],
         [ 1.0097,  1.6197, -1.0872, -0.2788, -0.6361,  0.9434, -1.3189,
          -0.2517],
         [ 0.8456,  1.5950, -1.1384, -0.5957, -0.8408,  1.2760, -0.3437,
          -0.7981],
         [ 0.9672,  1.9114, -0.8483, -0.3360, -0.6166,  0.5346, -1.3597,
          -0.2527],
         [ 0.8600,  1.1898, -0.6402, -0.4074, -0.9628,  1.5958, -1.2963,
          -0.3388],
         [ 0.5787,  1.5235,  0.2474, -0.6346, -1.0216,  1.2293, -1.4497,
          -0.4728],
         [ 0.8905,  1.1116, -0.6429, -0.5159, -1.1707,  1.5564, -1.2242,
          -0.0048],
         [ 0.2342,  1.6177, -1.1921, -0.4412,  0.1292,  1.2962, -1.3982,
          -0.2458],
         [ 0.9379,  1.5737, -0.9489, -0.0467, -0.6111,  0.9787, -1.5056,
          -0.3780],
         [ 0.9925,  1.5713, -0.9151, -0.354

### Stacking Encoder Layers with `TransformerEncoder`

To build a complete Transformer Encoder, we stack multiple `TransformerEncoderLayer` instances. The `TransformerEncoder` class will manage this stacking, allowing the input to pass through each layer sequentially. This enables the model to learn more complex relationships and representations from the input sequence.

In [322]:
class TransformerEncoder(nn.Module):
    def __init__(self, d_model, num_heads, ffn_hidden_dim, num_layers, dropout_rate=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, ffn_hidden_dim, dropout_rate)
            for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

In [323]:
# Demonstrate the TransformerEncoder
num_encoder_layers = 2 # Let's stack 2 encoder layers for demonstration

transformer_encoder = TransformerEncoder(
    d_model=d_model,
    num_heads=num_heads,
    ffn_hidden_dim=ffn_hidden_dim,
    num_layers=num_encoder_layers,
    dropout_rate=0.1
)

# Pass the original 'embedded' tensor through the stacked encoder
full_encoder_output = transformer_encoder(embedded)

print("\nShape of final output from TransformerEncoder:", full_encoder_output.shape)
print("Final output from TransformerEncoder (after multiple layers):\n", full_encoder_output)


Shape of final output from TransformerEncoder: torch.Size([1, 10, 8])
Final output from TransformerEncoder (after multiple layers):
 tensor([[[ 1.2219,  0.5246, -0.8733,  0.7729,  0.4440,  0.0533,  0.0180,
          -2.1614],
         [ 1.1940,  0.5047, -0.6360,  0.7643,  0.4081,  0.0642, -0.0278,
          -2.2715],
         [-0.0604,  0.6516, -0.4586,  1.1245,  0.6433,  0.2070,  0.2540,
          -2.3613],
         [ 1.1773,  0.4753, -0.3319,  0.7343,  0.3730,  0.0182, -0.0812,
          -2.3649],
         [ 0.9902,  0.4420, -0.6103,  0.8664,  0.4868,  0.1307,  0.0278,
          -2.3337],
         [ 1.3345, -0.4753, -0.3870,  0.8300,  0.5602,  0.2987,  0.0186,
          -2.1797],
         [ 0.9665,  0.5563, -0.6174,  0.8402,  0.4623,  0.1101,  0.0149,
          -2.3330],
         [ 1.2592,  0.4932, -0.5607,  0.8512,  0.4154,  0.2078, -0.5077,
          -2.1584],
         [ 1.2172,  0.3850, -0.6234,  0.7911,  0.4274,  0.0870, -0.0117,
          -2.2727],
         [ 1.2162,  0.3834, -

### Assembling the Full Transformer Encoder Model

Now, let's bring everything together into a single `Transformer` class. This class will encapsulate the input embedding, positional encoding, and the `TransformerEncoder` (which itself contains the `TransformerEncoderLayer`s). This creates a complete, end-to-end model that can take raw token IDs as input and produce the final processed embeddings from the encoder.

In [324]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ffn_hidden_dim, num_layers, max_len=100, dropout_rate=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len)
        self.encoder = TransformerEncoder(d_model, num_heads, ffn_hidden_dim, num_layers, dropout_rate)

    def forward(self, src):
        # 1. Apply embedding
        x = self.embedding(src)

        # 2. Add positional encoding
        x = self.positional_encoding(x)

        # 3. Pass through the Transformer Encoder
        x = self.encoder(x)

        return x

In [325]:
# Demonstrate the complete Transformer model

# Re-using existing parameters
# vocab_size, d_model, num_heads, ffn_hidden_dim, num_encoder_layers (num_layers)

full_transformer_model = Transformer(
    vocab_size=vocab_size, # Use the updated vocab_size
    d_model=d_model,
    num_heads=num_heads,
    ffn_hidden_dim=ffn_hidden_dim,
    num_layers=num_encoder_layers,
    max_len=max_seq_len, # Use the updated max_seq_len for consistency
    dropout_rate=0.1
)

# Pass the original tokenized input 'x' through the full model
# 'x' is torch.Size([1, 3]) with token IDs
final_model_output = full_transformer_model(x)

print("\nShape of final output from the complete Transformer model:", final_model_output.shape)
print("Final output from the complete Transformer model:\n", final_model_output)


Shape of final output from the complete Transformer model: torch.Size([1, 10, 8])
Final output from the complete Transformer model:
 tensor([[[-0.1770,  0.6037,  0.2479, -0.2981,  1.2092, -1.0124,  1.2630,
          -1.8362],
         [ 0.2169,  0.3100,  0.3717,  0.0713,  1.0522, -1.1654,  1.1339,
          -1.9906],
         [-0.1950,  0.3767,  0.4253, -0.3340,  1.2423, -1.0900,  1.3212,
          -1.7466],
         [-0.1531,  0.7461,  0.5367,  0.4179,  0.4528, -0.8866,  1.0780,
          -2.1918],
         [ 0.4945,  0.4541,  0.6398, -0.0534,  0.5463, -1.1979,  1.1416,
          -2.0250],
         [-0.1395,  0.4112,  0.2891, -0.2808,  1.2394, -1.0004,  1.3180,
          -1.8371],
         [-0.1379,  0.4129,  0.2848, -0.2818,  1.2395, -1.0014,  1.3195,
          -1.8357],
         [-0.1233,  0.5036,  0.4932, -0.2655,  1.3624, -0.8263,  0.8813,
          -2.0253],
         [-0.6058,  0.1832,  0.1257, -0.8658,  1.6727, -1.6687,  1.1179,
           0.0409],
         [-0.1605,  0.3892,  

### Adding a Classification Head

To use the `Transformer` encoder for a downstream task like text classification, we typically add a classification head. This head takes the output of the encoder (often the representation of the `[CLS]` token, or an averaged representation of the sequence) and maps it to the number of output classes using a linear layer.

In [326]:
class ClassificationHead(nn.Module):
    def __init__(self, d_model, num_classes):
        super().__init__()
        self.linear = nn.Linear(d_model, num_classes)

    def forward(self, encoder_output):
        # For simplicity, let's take the representation of the first token (like [CLS])
        # or you could average the sequence output: encoder_output.mean(dim=1)
        pooled_output = encoder_output[:, 0, :]
        logits = self.linear(pooled_output)
        return logits

In [327]:
class TransformerForClassification(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ffn_hidden_dim, num_layers, num_classes, max_len=100, dropout_rate=0.1):
        super().__init__()
        self.transformer_encoder = Transformer(
            vocab_size=vocab_size,
            d_model=d_model,
            num_heads=num_heads,
            ffn_hidden_dim=ffn_hidden_dim,
            num_layers=num_layers,
            max_len=max_len,
            dropout_rate=dropout_rate
        )
        self.classification_head = ClassificationHead(d_model, num_classes)

    def forward(self, src):
        encoder_output = self.transformer_encoder(src)
        logits = self.classification_head(encoder_output)
        return logits

In [328]:
# Demonstrate the TransformerForClassification model
num_classes = 2 # Example: binary classification (e.g., positive/negative sentiment)

full_transformer_for_classification_model = TransformerForClassification(
    vocab_size=vocab_size, # Use the updated vocab_size
    d_model=d_model,
    num_heads=num_heads,
    ffn_hidden_dim=ffn_hidden_dim,
    num_layers=num_encoder_layers,
    num_classes=num_classes,
    max_len=max_seq_len, # Use the updated max_seq_len for consistency
    dropout_rate=0.1
)

# Pass the original tokenized input 'x' through the full classification model
# 'x' is torch.Size([1, 3]) with token IDs
classification_output_logits = full_transformer_for_classification_model(x)

print("\nShape of output logits from the TransformerForClassification model:", classification_output_logits.shape)
print("Output logits from the TransformerForClassification model:\n", classification_output_logits)


Shape of output logits from the TransformerForClassification model: torch.Size([1, 2])
Output logits from the TransformerForClassification model:
 tensor([[-0.7958, -0.9421]], grad_fn=<AddmmBackward0>)


### Basic Training Setup

Now that you have a `TransformerForClassification` model, the next step is to train it on a dataset. This involves several key components:

1.  **Loss Function:** To quantify how far off your model's predictions are from the actual labels.
2.  **Optimizer:** To adjust the model's parameters based on the calculated loss.
3.  **Training Loop:** The iterative process of feeding data to the model, calculating loss, and updating parameters.

In [329]:
# 1. Define Loss Function and Optimizer
loss_fn = nn.CrossEntropyLoss() # Suitable for classification tasks (predicting the next word)
# The optimizer should be re-initialized with the parameters of the new model
optimizer = torch.optim.Adam(full_transformer_for_classification_model.parameters(), lr=0.001)

### Training Loop for Next Word Prediction

Now, let's train our Transformer model for the next word prediction task using the `NextWordDataset` and `next_word_dataloader`.

In [330]:
from torch.utils.data import Dataset, DataLoader

class NextWordDataset(Dataset):
    def __init__(self, df, vocab, max_seq_len):
        self.data = []
        for _, row in df.iterrows():
            # Tokenize input sequence
            input_ids = tokenize(row['input_sequence_text'], vocab, max_seq_len).squeeze(0)
            # Target is the word ID
            target_id = row['target_word']
            self.data.append((input_ids, torch.tensor(target_id, dtype=torch.long)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# Initialize the dataloader using existing next_word_df and new_vocab
next_word_dataset = NextWordDataset(next_word_df, new_vocab, max_seq_len)
next_word_dataloader = DataLoader(next_word_dataset, batch_size=2, shuffle=True)

In [331]:
epochs = 10

print("\n--- Starting Batched Training Loop for Next Word Prediction ---")

# Ensure we use the model initialized for the full vocabulary (37 classes)
optimizer = torch.optim.Adam(full_transformer_for_next_word_model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(epochs):
    full_transformer_for_next_word_model.train()
    total_loss = 0
    correct_predictions = 0
    total_samples = 0

    for batch_idx, (input_ids_batch, target_word_ids_batch) in enumerate(next_word_dataloader):
        optimizer.zero_grad()

        # Forward pass
        output_logits = full_transformer_for_next_word_model(input_ids_batch)

        # Calculate loss
        loss = loss_fn(output_logits, target_word_ids_batch)

        # Backward pass
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Calculate Accuracy
        predictions = torch.argmax(output_logits, dim=-1)
        correct_predictions += (predictions == target_word_ids_batch).sum().item()
        total_samples += target_word_ids_batch.size(0)

    avg_loss = total_loss / len(next_word_dataloader)
    accuracy = correct_predictions / total_samples
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2%}")

print("--- Batched Training Loop Finished ---")

# Set to evaluation mode
full_transformer_for_next_word_model.eval()


--- Starting Batched Training Loop for Next Word Prediction ---
Epoch 1/10, Loss: 2.9686, Accuracy: 13.89%
Epoch 2/10, Loss: 2.9592, Accuracy: 16.67%
Epoch 3/10, Loss: 2.9259, Accuracy: 16.67%
Epoch 4/10, Loss: 2.8626, Accuracy: 11.11%
Epoch 5/10, Loss: 2.8196, Accuracy: 22.22%
Epoch 6/10, Loss: 2.8176, Accuracy: 22.22%
Epoch 7/10, Loss: 2.8134, Accuracy: 27.78%
Epoch 8/10, Loss: 2.7511, Accuracy: 16.67%
Epoch 9/10, Loss: 2.7646, Accuracy: 22.22%
Epoch 10/10, Loss: 2.7540, Accuracy: 25.00%
--- Batched Training Loop Finished ---


TransformerForClassification(
  (transformer_encoder): Transformer(
    (embedding): Embedding(37, 8)
    (positional_encoding): PositionalEncoding()
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (Wq): Linear(in_features=8, out_features=8, bias=True)
          (Wk): Linear(in_features=8, out_features=8, bias=True)
          (Wv): Linear(in_features=8, out_features=8, bias=True)
          (Wo): Linear(in_features=8, out_features=8, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (layer_norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
          (ffn): Sequential(
            (0): Linear(in_features=8, out_features=32, bias=True)
            (1): ReLU()
            (2): Linear(in_features=32, out_features=8, bias=True)
          )
          (ffn_dropout): Dropout(p=0.1, inplace=False)
          (layer_norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
        )
      )
 

In [335]:
import torch
from torch.utils.data import random_split

# Split dataset into training and testing (80% train, 20% test)
train_size = int(0.8 * len(next_word_dataset))
test_size = len(next_word_dataset) - train_size
train_subset, test_subset = random_split(next_word_dataset, [train_size, test_size])

train_loader = DataLoader(train_subset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=2, shuffle=False)

# Re-initialize the model
full_transformer_for_next_word_model = TransformerForClassification(
    vocab_size=vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    ffn_hidden_dim=ffn_hidden_dim,
    num_layers=num_encoder_layers,
    num_classes=vocab_size,
    max_len=max_seq_len,
    dropout_rate=0.1
)

optimizer = torch.optim.Adam(full_transformer_for_next_word_model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

epochs = 10
print("--- Starting Training with Accuracy and Testing ---")

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    full_transformer_for_next_word_model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for inputs, targets in train_loader:
        optimizer.zero_grad()
        logits = full_transformer_for_next_word_model(inputs)
        loss = loss_fn(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = torch.argmax(logits, dim=-1)
        train_correct += (preds == targets).sum().item()
        train_total += targets.size(0)

    # --- TESTING/EVALUATION PHASE ---
    full_transformer_for_next_word_model.eval()
    test_loss = 0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for inputs, targets in test_loader:
            logits = full_transformer_for_next_word_model(inputs)
            loss = loss_fn(logits, targets)
            test_loss += loss.item()
            preds = torch.argmax(logits, dim=-1)
            test_correct += (preds == targets).sum().item()
            test_total += targets.size(0)

    print(f"Epoch {epoch+1}/{epochs}:")
    print(f"  [Train] Loss: {train_loss/len(train_loader):.4f}, Acc: {100*train_correct/train_total:.2f}%")
    print(f"  [Test]  Loss: {test_loss/len(test_loader):.4f}, Acc: {100*test_correct/test_total:.2f}%")

print("--- Training Finished ---")

--- Starting Training with Accuracy and Testing ---
Epoch 1/10:
  [Train] Loss: 3.8708, Acc: 3.57%
  [Test]  Loss: 3.6798, Acc: 0.00%
Epoch 2/10:
  [Train] Loss: 3.7549, Acc: 3.57%
  [Test]  Loss: 3.6914, Acc: 0.00%
Epoch 3/10:
  [Train] Loss: 3.6807, Acc: 3.57%
  [Test]  Loss: 3.7605, Acc: 0.00%
Epoch 4/10:
  [Train] Loss: 3.6294, Acc: 7.14%
  [Test]  Loss: 3.8072, Acc: 0.00%
Epoch 5/10:
  [Train] Loss: 3.6284, Acc: 3.57%
  [Test]  Loss: 3.8712, Acc: 0.00%
Epoch 6/10:
  [Train] Loss: 3.6210, Acc: 3.57%
  [Test]  Loss: 3.9129, Acc: 0.00%
Epoch 7/10:
  [Train] Loss: 3.5326, Acc: 7.14%
  [Test]  Loss: 3.9282, Acc: 0.00%
Epoch 8/10:
  [Train] Loss: 3.3864, Acc: 14.29%
  [Test]  Loss: 4.0225, Acc: 12.50%
Epoch 9/10:
  [Train] Loss: 3.4721, Acc: 0.00%
  [Test]  Loss: 4.0229, Acc: 12.50%
Epoch 10/10:
  [Train] Loss: 3.3786, Acc: 7.14%
  [Test]  Loss: 4.0679, Acc: 12.50%
--- Training Finished ---


### Making Next Word Predictions

Now that the model is trained, let's create a function to predict the next word given an input sequence.

In [333]:
def predict_next_word(input_text, model, vocab, idx_to_word, max_seq_len):
    model.eval() # Set model to evaluation mode

    # Tokenize the input text
    input_ids = tokenize(input_text, vocab, max_seq_len)

    # Make prediction
    with torch.no_grad():
        output_logits = model(input_ids)

        # Get probabilities over the vocabulary
        probabilities = torch.softmax(output_logits, dim=-1)

        # Get the most likely next word ID
        predicted_word_id = torch.argmax(probabilities, dim=-1).item()

    predicted_word = idx_to_word.get(predicted_word_id, '<unk>')
    return predicted_word, probabilities.tolist()[0]

# Example usage:
# We use 'full_transformer_for_next_word_model' and 'new_vocab' which contain the trained weights and full word list
input_sequence = "i am anugrah"
predicted_word, probs = predict_next_word(input_sequence, full_transformer_for_next_word_model, new_vocab, idx_to_word, max_seq_len)

print(f"Input sequence: '{input_sequence}'")
print(f"Predicted next word: '{predicted_word}'")
print(f"Probabilities (top 5): {sorted(probs, reverse=True)[:5]}")

input_sequence_2 = "this is a positive"
predicted_word_2, probs_2 = predict_next_word(input_sequence_2, full_transformer_for_next_word_model, new_vocab, idx_to_word, max_seq_len)

print(f"\nInput sequence: '{input_sequence_2}'")
print(f"Predicted next word: '{predicted_word_2}'")
print(f"Probabilities (top 5): {sorted(probs_2, reverse=True)[:5]}")

Input sequence: 'i am anugrah'
Predicted next word: 'this'
Probabilities (top 5): [0.0904940739274025, 0.06865762919187546, 0.06563367694616318, 0.061420660465955734, 0.055593568831682205]

Input sequence: 'this is a positive'
Predicted next word: 'am'
Probabilities (top 5): [0.07130128145217896, 0.06420949101448059, 0.0611000657081604, 0.04600774124264717, 0.045478641986846924]


In [334]:
# Fix the prediction example usage with correct variables
input_sequence = "i am anugrah"
# Using 'new_vocab' as it was used in the dataset and contains all necessary tokens
predicted_word, probs = predict_next_word(input_sequence, full_transformer_for_next_word_model, new_vocab, idx_to_word, max_seq_len)

print(f"Input sequence: '{input_sequence}'")
print(f"Predicted next word: '{predicted_word}'")
print(f"Probabilities (top 5): {sorted(probs, reverse=True)[:5]}")

Input sequence: 'i am anugrah'
Predicted next word: 'this'
Probabilities (top 5): [0.0904940739274025, 0.06865762919187546, 0.06563367694616318, 0.061420660465955734, 0.055593568831682205]
